In [3]:
"""Build biological drug profiles and pairwise similarity matrices.

This module is designed for local, pre-existing data structures such as pandas
DataFrames and lookup dictionaries. It does not query external APIs.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from itertools import combinations
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, as_completed



In [4]:


# ================================================================================
# BIOLOGICAL PROFILE SIMILARITY & DIRECTIONAL CONTAINMENT PIPELINE
# ================================================================================

# OVERVIEW:
# This pipeline processes pre-paired drug biological profile data stored in a 
# pandas DataFrame or Parquet file. It computes both symmetric biological overlap 
# and directional (asymmetric) profile containment across five distinct biological 
# feature spaces.

# MATHEMATICAL FRAMEWORK:
# 1. Jaccard Similarity (|A ∩ B| / |A ∪ B|):
#    - Measures overall symmetric shared biological footprint relative to the total 
#      combined footprint of both drugs.

# 2. Asymmetric Tversky Index (|A ∩ B| / (|A ∩ B| + α|A \ B| + β|B \ A|)):
#    - Configured with α = 1.0, β = 0.0 to evaluate directional containment.
#    - Evaluates what fraction of one drug's total biological profile is completely 
#      subsumed within the second drug's profile.
#    - Resolves profile size asymmetry (e.g., preventing a narrow 1-target drug from 
#      having its high-confidence overlap diluted when compared against a broad 
#      50-target drug).

# FEATURE LAYERS EVALUATED:
# - Identity ('target_other_ids'): Direct UniProt / target identifier overlap.
# - Gene Ontology ('target_go_terms'): Shared functional, process, and localization context.
# - Pfam Domains ('target_pfam_domains'): Mechanistic domain architecture similarity 
#   (detects shared family mechanisms even when exact targets differ).
# - Pathways ('target_pathway_neighbors'): Shared functional network neighborhood across 
#   expanded pathway systems.
# - FASTA Sequences ('target_fasta_sequences'): Direct target sequence identity matching.

# OUTPUT METRICS GENERATED PER FEATURE PAIR:
# - `<prefix>_jaccard`: Raw symmetric overlap (0.0 to 1.0)
# - `<prefix>_tversky_1_in_2`: Containment fraction of Drug 1 inside Drug 2 (0.0 to 1.0)
# - `<prefix>_tversky_2_in_1`: Containment fraction of Drug 2 inside Drug 1 (0.0 to 1.0)
# ================================================================================


In [5]:
"""
================================================================================
BATCHED & MULTI-CORE BIOLOGICAL PROFILE SIMILARITY PIPELINE
================================================================================
This version splits large DataFrames into chunks and processes them in parallel 
across CPU cores to accelerate execution on large datasets.
================================================================================
"""

import os
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

# ==========================================
# 1. SET HELPER & SIMILARITY METRICS
# ==========================================

def _to_set(val):
    """Safely converts DataFrame cell values to sets for set-math operations."""
    if isinstance(val, set):
        return val
    if isinstance(val, (list, tuple, np.ndarray, pd.Series)):
        return set(x for x in val if x is not None and str(x).lower() != 'nan')
        
    if pd.isna(val):
        return set()
        
    if isinstance(val, str):
        return set(x.strip() for x in val.split(',') if x.strip())
        
    return set([val])


def compute_jaccard(set_a, set_b):
    """Computes standard Jaccard Similarity: |A ∩ B| / |A ∪ B|."""
    if not set_a and not set_b:
        return 0.0
    intersection = len(set_a.intersection(set_b))
    union = len(set_a.union(set_b))
    return intersection / union if union > 0 else 0.0


def compute_tversky(set_a, set_b, alpha=1.0, beta=0.0):
    """Computes asymmetric Tversky Index (defaults to directional containment)."""
    if not set_a and not set_b:
        return 0.0
    
    intersection = len(set_a.intersection(set_b))
    only_a = len(set_a - set_b)
    only_b = len(set_b - set_a)
    
    denominator = intersection + (alpha * only_a) + (beta * only_b)
    return intersection / denominator if denominator > 0 else 0.0


# ==========================================
# 2. BATCH WORKER FUNCTION
# ==========================================

def _process_chunk(args):
    """
    Worker function executed in parallel processes.
    Computes metrics for a single chunk of the DataFrame.
    """
    chunk_df, feature_mappings = args
    result_dfs = []

    for col1, col2, prefix in feature_mappings:
        if col1 in chunk_df.columns and col2 in chunk_df.columns:
            sets_a = chunk_df[col1].apply(_to_set)
            sets_b = chunk_df[col2].apply(_to_set)

            jaccard_scores = [compute_jaccard(a, b) for a, b in zip(sets_a, sets_b)]
            tversky_a_in_b = [compute_tversky(a, b, alpha=1.0, beta=0.0) for a, b in zip(sets_a, sets_b)]
            tversky_b_in_a = [compute_tversky(b, a, alpha=1.0, beta=0.0) for a, b in zip(sets_a, sets_b)]

            metrics_df = pd.DataFrame({
                f"{prefix}_jaccard": jaccard_scores,
                f"{prefix}_tversky_1_in_2": tversky_a_in_b,
                f"{prefix}_tversky_2_in_1": tversky_b_in_a
            }, index=chunk_df.index)

            result_dfs.append(metrics_df)

    if result_dfs:
        return pd.concat(result_dfs, axis=1)
    return pd.DataFrame(index=chunk_df.index)


# ==========================================
# 3. BATCHED PIPELINE EXECUTION
# ==========================================

def process_biological_profiles_batched(df, feature_mappings, batch_size=25000, n_jobs=4):
    """
    Robust Windows + Jupyter multiprocessing using joblib.
    """
    total_rows = len(df)
    chunks = [df.iloc[i : i + batch_size] for i in range(0, total_rows, batch_size)]
    
    print(f"Starting pipeline...")
    print(f"Total Rows: {total_rows:,} | Batch Size: {batch_size:,} | Workers: {n_jobs} | Batches: {len(chunks)}")

    # joblib uses 'loky' by default, which safely spawns processes in Jupyter on Windows
    results = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(_process_chunk)((chunk, feature_mappings))
        for chunk in tqdm(chunks, desc="Processing Batches")
    )


    print("\nMerging computed metrics back to original dataset...")
    all_metrics = pd.concat(results, axis=0)
    final_df = pd.concat([df, all_metrics], axis=1)
    print("Batched processing complete.")
    
    return final_df
# ==========================================
# 4. HOW TO RUN
# ==========================================
if __name__ == "__main__":

    my_feature_columns = [
        ('target_other_ids_1', 'target_other_ids_2', 'identity'),          
        ('target_go_terms_1', 'target_go_terms_2', 'go'),                  
        ('target_pfam_domains_1', 'target_pfam_domains_2', 'pfam'),        
        ('target_pathway_neighbors_1', 'target_pathway_neighbors_2', 'pathway'), 
        ('target_fasta_sequences_1', 'target_fasta_sequences_2', 'fasta')  
    ]

    # Run on your DataFrames:
    # adverse_biological_overlap_df = process_biological_profiles_batched(
    #     adverse_pairs_df, 
    #     my_feature_columns, 
    #     batch_size=25000, 
    #     n_jobs=-1
    # )
    pass

In [6]:
non_interacting_pairs_df = pd.read_parquet("C:\\Users\\ashto\\OneDrive - Eastern Connecticut State University\\Project 5. Data\\processed\\a_potential_final_df\\non_interacting_pairs_final.parquet")
adverse_pairs_df = pd.read_parquet("C:\\Users\\ashto\\OneDrive - Eastern Connecticut State University\\Project 5. Data\\processed\\a_potential_final_df\\adverse_final_df.parquet")
synergistic_pairs_df = pd.read_csv("C:\\Users\\ashto\\OneDrive - Eastern Connecticut State University\\Project 5. Data\\raw\\removal_of_positive_ddis\\positive_non_adverse_15971_ddis_conservative.csv")

In [7]:
# `target_other_ids` is only a sparse fallback identifier (populated when a drug has
# no primary UniProt mapping), so using it alone leaves 'identity' near-constant 0 for
# the majority of pairs that already have `target_uniprot_ids`. Build a combined
# identity set per drug (union of both) so neither source is silently dropped.
def _union_target_ids(df, suffix):
    return [_to_set(u) | _to_set(o) for u, o in zip(df[f'target_uniprot_ids{suffix}'], df[f'target_other_ids{suffix}'])]

for _df in (adverse_pairs_df, non_interacting_pairs_df):
    _df['target_identity_ids_1'] = _union_target_ids(_df, '_1')
    _df['target_identity_ids_2'] = _union_target_ids(_df, '_2')

my_feature_columns = [
        ('target_identity_ids_1', 'target_identity_ids_2', 'identity'),          
        ('target_go_terms_1', 'target_go_terms_2', 'go'),                  
        ('target_pfam_domains_1', 'target_pfam_domains_2', 'pfam'),        
        ('target_pathway_neighbors_1', 'target_pathway_neighbors_2', 'pathway'), 
        ('target_fasta_sequences_1', 'target_fasta_sequences_2', 'fasta')  
    ]

In [8]:
adverse_biological_overlap_df = process_biological_profiles_batched(adverse_pairs_df, my_feature_columns).drop(columns=['target_identity_ids_1', 'target_identity_ids_2'])
non_interacting_biological_overlap_df = process_biological_profiles_batched(non_interacting_pairs_df, my_feature_columns).drop(columns=['target_identity_ids_1', 'target_identity_ids_2'])


Starting pipeline...
Total Rows: 478,324 | Batch Size: 25,000 | Workers: 4 | Batches: 20


Processing Batches:   0%|          | 0/20 [00:00<?, ?it/s]


Merging computed metrics back to original dataset...
Batched processing complete.
Starting pipeline...
Total Rows: 162,893 | Batch Size: 25,000 | Workers: 4 | Batches: 7


Processing Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Merging computed metrics back to original dataset...
Batched processing complete.


In [9]:

print(adverse_biological_overlap_df.columns.tolist())

['drug1_id', 'drug1_name', 'drug2_id', 'drug2_name', 'description', 'pair_key', 'smiles_1', 'smiles_2', 'atc_codes_list_1', 'atc_codes_list_2', 'target_uniprot_ids_1', 'target_uniprot_ids_2', 'target_fasta_sequences_1', 'target_fasta_sequences_2', 'target_other_ids_1', 'target_other_ids_2', 'target_go_terms_1', 'target_go_terms_2', 'target_pfam_domains_1', 'target_pfam_domains_2', 'target_pathway_neighbors_1', 'target_pathway_neighbors_2', 'identity_jaccard', 'identity_tversky_1_in_2', 'identity_tversky_2_in_1', 'go_jaccard', 'go_tversky_1_in_2', 'go_tversky_2_in_1', 'pfam_jaccard', 'pfam_tversky_1_in_2', 'pfam_tversky_2_in_1', 'pathway_jaccard', 'pathway_tversky_1_in_2', 'pathway_tversky_2_in_1', 'fasta_jaccard', 'fasta_tversky_1_in_2', 'fasta_tversky_2_in_1']


In [10]:
drugs_with_only_other_ids = ((adverse_pairs_df['target_uniprot_ids_1'].apply(lambda x: len(_to_set(x))) == 0)
                              & (adverse_pairs_df['target_other_ids_1'].apply(lambda x: len(_to_set(x))) > 0)).sum()
print(f"Adverse rows where drug1 has ONLY target_other_ids (no uniprot ids): {drugs_with_only_other_ids:,} / {len(adverse_pairs_df):,}")

for col in ['identity_jaccard', 'go_jaccard', 'pfam_jaccard', 'pathway_jaccard', 'fasta_jaccard']:
    nonzero_pct = (adverse_biological_overlap_df[col] > 0).mean() * 100
    print(f"adverse  {col:<16}: {nonzero_pct:5.1f}% of pairs are nonzero")
print()
for col in ['identity_jaccard', 'go_jaccard', 'pfam_jaccard', 'pathway_jaccard', 'fasta_jaccard']:
    nonzero_pct = (non_interacting_biological_overlap_df[col] > 0).mean() * 100
    print(f"non_int  {col:<16}: {nonzero_pct:5.1f}% of pairs are nonzero")


Adverse rows where drug1 has ONLY target_other_ids (no uniprot ids): 10,041 / 478,324
adverse  identity_jaccard:   6.2% of pairs are nonzero
adverse  go_jaccard      :   3.7% of pairs are nonzero
adverse  pfam_jaccard    :   1.4% of pairs are nonzero
adverse  pathway_jaccard :  78.9% of pairs are nonzero
adverse  fasta_jaccard   :   5.9% of pairs are nonzero

non_int  identity_jaccard:   1.0% of pairs are nonzero
non_int  go_jaccard      :   1.8% of pairs are nonzero
non_int  pfam_jaccard    :   0.2% of pairs are nonzero
non_int  pathway_jaccard :  67.9% of pairs are nonzero
non_int  fasta_jaccard   :   0.9% of pairs are nonzero


## Extended biological profile: T/E/R/C roles, L0/L1/L2 ladder, GO-MF/BP/CC, native pathways

Rebuilds the per-drug biological profile from the raw DrugBank XML plus ChEMBL to match the
formal spec: pools `T(d) = T^DB ∪ T^mech ∪ T^bio` for every drug (not just gap-filling), adds the
`E(d)`/`R(d)`/`C(d)` protein roles to form the `L0 ⊆ L1 ⊆ L2` ladder, splits GO by aspect
(MF/BP/CC), and adds the native pathway profile (`Φ_native`) alongside the existing inferred one.


In [11]:
import sys
SRC_FOLDER = r"C:\Users\ashto\ddi-prediction\src_test"
if SRC_FOLDER not in sys.path:
    sys.path.append(SRC_FOLDER)

from smpdb_protein_pathway import (
    parse_drugbank_biological_profiles,
    get_targets_from_chembl_batch,
    PathwayMapper,
)

DRUGBANK_XML_PATH = r"C:\Users\ashto\ddi-prediction\data\raw\drugbank_full_database_V5.1.14.zip"
SMPDB_ZIP = r"C:\Users\ashto\ddi-prediction\data\smpdb_pathways_data_csv\smpdb_proteins.csv.zip"

wanted_drug_ids = set(adverse_pairs_df['drug1_id']) | set(adverse_pairs_df['drug2_id']) \
    | set(non_interacting_pairs_df['drug1_id']) | set(non_interacting_pairs_df['drug2_id'])
print(f"Unique drugs across both pair sets: {len(wanted_drug_ids):,}")


Unique drugs across both pair sets: 1,907


In [12]:
import time

_t0 = time.time()
drug_profiles = parse_drugbank_biological_profiles(DRUGBANK_XML_PATH, wanted_ids=wanted_drug_ids)
print(f"Parsed {len(drug_profiles):,} / {len(wanted_drug_ids):,} wanted drugs in {time.time() - _t0:.1f}s")

missing = wanted_drug_ids - set(drug_profiles.keys())
print(f"Drugs not found in DrugBank XML at all: {len(missing):,}")


Parsed 1,907 / 1,907 wanted drugs in 237.8s
Drugs not found in DrugBank XML at all: 0


In [13]:
# Always-pool T(d) = T^DB ∪ T^mech ∪ T^bio for every drug (not just drugs missing DrugBank targets).
all_chembl_ids = sorted({p['chembl_id'] for p in drug_profiles.values() if p['chembl_id']})
print(f"Drugs with a ChEMBL id: {len(all_chembl_ids):,} / {len(drug_profiles):,}")

_t0 = time.time()
chembl_targets_by_id = get_targets_from_chembl_batch(all_chembl_ids, min_pchembl=6.0)
print(f"Resolved ChEMBL targets (mechanism + bioactivity) in {time.time() - _t0:.1f}s")


Drugs with a ChEMBL id: 1,845 / 1,907
Resolved ChEMBL targets (mechanism + bioactivity) in 1.6s


In [14]:
# Build the L0 ⊆ L1 ⊆ L2 protein-role ladder and the native pathway profile (Φ_native) per drug.
for dbid, p in drug_profiles.items():
    t_db = set(p['target_uniprot_ids'])
    t_chembl = set(chembl_targets_by_id.get(p['chembl_id'], [])) if p['chembl_id'] else set()
    L0 = t_db | t_chembl                                    # T(d) = T^DB ∪ T^mech ∪ T^bio
    L1 = L0 | set(p['enzyme_uniprot_ids']) | set(p['transporter_uniprot_ids'])
    L2 = L1 | set(p['carrier_uniprot_ids'])
    p['bio_L0'] = L0
    p['bio_L1'] = L1
    p['bio_L2'] = L2

print("Sample L0/L1/L2 sizes (first 5 drugs):")
for dbid in list(drug_profiles)[:5]:
    p = drug_profiles[dbid]
    print(f"  {dbid}: L0={len(p['bio_L0'])}  L1={len(p['bio_L1'])}  L2={len(p['bio_L2'])}")


Sample L0/L1/L2 sizes (first 5 drugs):
  DB00006: L0=1  L1=2  L2=2
  DB00014: L0=3  L1=3  L2=3
  DB00027: L0=1  L1=2  L2=2
  DB00035: L0=4  L1=6  L2=6
  DB00080: L0=1  L1=2  L2=8


In [15]:
# Build ONE PathwayMapper (SMPDB canonical + DrugBank supplemental) and reuse it for both
# the native pathway profile (Φ_native) and the re-seeded inferred neighbor expansion (Φ_infer).
_t0 = time.time()
pathway_mapper = PathwayMapper(xml_path=DRUGBANK_XML_PATH, smpdb_protein_zip=SMPDB_ZIP)
print(f"PathwayMapper built in {time.time() - _t0:.1f}s")

all_native_pathway_ids = sorted({pid for p in drug_profiles.values() for pid in p['native_pathway_ids']})
print(f"Distinct native pathway ids across all drugs: {len(all_native_pathway_ids):,}")

pathway_id_to_proteins = pathway_mapper.get_proteins_by_pathway_batch(all_native_pathway_ids)

for p in drug_profiles.values():
    proteins = set()
    for pid in p['native_pathway_ids']:
        proteins.update(pathway_id_to_proteins.get(pid, []))
    p['native_pathway_proteins'] = proteins   # Φ_native(d)


[SMPDB] Found 48687 pathway CSV files
[SMPDB] Loaded 48642 pathways
[SMPDB] Loaded 1489 proteins
[SMPDB] Loaded 303912 pathway-protein edges
[DrugBank] ZIP detected → loading: drugbank_full_database_V5.1.14.xml
PathwayMapper built in 510.8s
Distinct native pathway ids across all drugs: 2,606


In [16]:
# Re-seed the existing inferred pathway-neighbor expansion (Φ_infer) from the newly pooled
# L0(d) = T(d) (was previously seeded from T^DB(d) alone).
all_L0_proteins = sorted({prot for p in drug_profiles.values() for prot in p['bio_L0']})
protein_to_pathways = pathway_mapper.get_pathways_by_protein_batch(all_L0_proteins)

all_pathways_from_L0 = sorted({pw for pws in protein_to_pathways.values() for pw in pws})
pathway_to_proteins_for_infer = pathway_mapper.get_proteins_by_pathway_batch(all_pathways_from_L0)

for p in drug_profiles.values():
    drug_pathways = set()
    for prot in p['bio_L0']:
        drug_pathways.update(protein_to_pathways.get(prot, []))
    drug_neighbors = set()
    for pw in drug_pathways:
        drug_neighbors.update(pathway_to_proteins_for_infer.get(pw, []))
    p['inferred_pathway_neighbors'] = drug_neighbors   # Φ_infer(d; L0)

print("Sample Φ_native / Φ_infer sizes (first 5 drugs):")
for dbid in list(drug_profiles)[:5]:
    p = drug_profiles[dbid]
    print(f"  {dbid}: native={len(p['native_pathway_proteins'])}  inferred={len(p['inferred_pathway_neighbors'])}")


Sample Φ_native / Φ_infer sizes (first 5 drugs):
  DB00006: native=21  inferred=1305
  DB00014: native=0  inferred=383
  DB00027: native=0  inferred=0
  DB00035: native=0  inferred=492
  DB00080: native=0  inferred=0


In [ ]:
import numpy as np

# Compute scores via direct drugbank_id -> feature-set lookup instead of duplicating each
# drug's (sometimes 1000+ element) profile sets across ~640K pair rows -- the previous
# per-row column attach approach exhausted memory (478K+163K rows x large sets x 9 layers).
_profile_cols = ['bio_L0', 'bio_L1', 'bio_L2', 'target_go_mf', 'target_go_bp', 'target_go_cc',
                  'target_pfam_domains', 'native_pathway_proteins', 'inferred_pathway_neighbors']

_feature_sets_by_drug = {
    dbid: {col: (p[col] if isinstance(p[col], set) else set(p[col])) for col in _profile_cols}
    for dbid, p in drug_profiles.items()
}


def compute_layer_similarity(pairs_df, feature_sets_by_drug, layer_specs):
    """layer_specs: list of (profile_key, prefix). Returns scores plus, per pair, the actual
    shared (intersection) ids -- the full per-drug id sets are NOT duplicated per row (that's
    what blew up memory before); use build_drug_profile_lookup for the full per-drug id lists."""
    empty = set()
    d1 = pairs_df['drug1_id'].to_numpy()
    d2 = pairs_df['drug2_id'].to_numpy()
    n = len(pairs_df)
    out = {}
    for profile_key, prefix in layer_specs:
        jacc = np.empty(n)
        t12 = np.empty(n)
        t21 = np.empty(n)
        shared_ids = [None] * n
        for i in range(n):
            a = feature_sets_by_drug.get(d1[i], {}).get(profile_key, empty)
            b = feature_sets_by_drug.get(d2[i], {}).get(profile_key, empty)
            jacc[i] = compute_jaccard(a, b)
            t12[i] = compute_tversky(a, b)
            t21[i] = compute_tversky(b, a)
            shared_ids[i] = sorted(a & b)
        out[f'{prefix}_jaccard'] = jacc
        out[f'{prefix}_tversky_1_in_2'] = t12
        out[f'{prefix}_tversky_2_in_1'] = t21
        out[f'{prefix}_shared_ids'] = shared_ids
    return pd.DataFrame(out, index=pairs_df.index)


def build_drug_profile_lookup(feature_sets_by_drug, layer_specs):
    """One row per drug with the actual id list for each layer (bio_L0/L1/L2, GO, Pfam,
    pathways). Join on `drugbank_id` to recover the full per-drug ids behind any pair's scores."""
    rows = []
    for dbid, sets_by_key in feature_sets_by_drug.items():
        row = {'drugbank_id': dbid}
        for profile_key, prefix in layer_specs:
            row[f'{prefix}_ids'] = sorted(sets_by_key.get(profile_key, set()))
        rows.append(row)
    return pd.DataFrame(rows)



In [ ]:

# Extended feature set: identity at each ladder level (L0/L1/L2), GO split by aspect (MF/BP/CC),
# Pfam (now DrugBank-sourced, still T(d)-only per spec §3.1), and both pathway flavors
# (native = Φ_native, inferred = Φ_infer reseeded from pooled L0).
layer_specs = [
    ('bio_L0', 'identity_L0'),
    ('bio_L1', 'identity_L1'),
    ('bio_L2', 'identity_L2'),
    ('target_go_mf', 'go_mf'),
    ('target_go_bp', 'go_bp'),
    ('target_go_cc', 'go_cc'),
    ('target_pfam_domains', 'pfam'),
    ('native_pathway_proteins', 'pathway_native'),
    ('inferred_pathway_neighbors', 'pathway_inferred'),
]

adverse_extended_df = pd.concat(
    [adverse_pairs_df[['drug1_id', 'drug2_id']], compute_layer_similarity(adverse_pairs_df, _feature_sets_by_drug, layer_specs)],
    axis=1,
)
non_interacting_extended_df = pd.concat(
    [non_interacting_pairs_df[['drug1_id', 'drug2_id']], compute_layer_similarity(non_interacting_pairs_df, _feature_sets_by_drug, layer_specs)],
    axis=1,
)
print(adverse_extended_df.shape, non_interacting_extended_df.shape)

# Full per-drug id lists (one row per unique drug, not per pair) so the actual pfam/pathway/GO
# ids behind any pair's scores can be looked up via `drugbank_id` without duplicating them.
drug_profile_lookup_df = build_drug_profile_lookup(_feature_sets_by_drug, layer_specs)
print(drug_profile_lookup_df.shape)



(478324, 29) (162893, 29)


In [ ]:
adverse_extended_df.to_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h1_biological_overlap\adverse_biological_overlap_extended.parquet")
non_interacting_extended_df.to_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h1_biological_overlap\non_interacting_biological_overlap_extended.parquet")
drug_profile_lookup_df.to_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h1_biological_overlap\drug_biological_profile_ids.parquet")
print("Saved extended biological similarity tables and per-drug id lookup table.")


Saved extended biological similarity tables.


In [19]:
# Verify the ladder is monotone (jaccard should generally rise L0 -> L1 -> L2 as sets grow)
# and check nonzero coverage per layer for both pair sets.
for label, df in (('adverse', adverse_extended_df), ('non_interacting', non_interacting_extended_df)):
    print(f"=== {label} ===")
    for prefix in ['identity_L0', 'identity_L1', 'identity_L2', 'go_mf', 'go_bp', 'go_cc',
                   'pfam', 'pathway_native', 'pathway_inferred']:
        col = f'{prefix}_jaccard'
        nonzero_pct = (df[col] > 0).mean() * 100
        mean_val = df[col].mean()
        print(f"  {prefix:<18}: {nonzero_pct:5.1f}% nonzero | mean jaccard = {mean_val:.4f}")
    print()


=== adverse ===
  identity_L0       :   6.1% nonzero | mean jaccard = 0.0158
  identity_L1       :  50.7% nonzero | mean jaccard = 0.0616
  identity_L2       :  52.5% nonzero | mean jaccard = 0.0619
  go_mf             :  30.5% nonzero | mean jaccard = 0.0361
  go_bp             :  39.4% nonzero | mean jaccard = 0.0281
  go_cc             :  68.8% nonzero | mean jaccard = 0.0981
  pfam              :  18.5% nonzero | mean jaccard = 0.0750
  pathway_native    :   4.6% nonzero | mean jaccard = 0.0065
  pathway_inferred  :  77.1% nonzero | mean jaccard = 0.1223

=== non_interacting ===
  identity_L0       :   0.9% nonzero | mean jaccard = 0.0023
  identity_L1       :  17.0% nonzero | mean jaccard = 0.0177
  identity_L2       :  18.8% nonzero | mean jaccard = 0.0185
  go_mf             :  18.8% nonzero | mean jaccard = 0.0156
  go_bp             :  23.8% nonzero | mean jaccard = 0.0088
  go_cc             :  54.6% nonzero | mean jaccard = 0.0687
  pfam              :   7.7% nonzero | mean 

In [20]:
adverse_extended_df.head()


,drug1_id,drug2_id,identity_L0_jaccard,identity_L0_tversky_1_in_2,identity_L0_tversky_2_in_1,identity_L1_jaccard,identity_L1_tversky_1_in_2,identity_L1_tversky_2_in_1,identity_L2_jaccard,identity_L2_tversky_1_in_2,...,go_cc_tversky_2_in_1,pfam_jaccard,pfam_tversky_1_in_2,pfam_tversky_2_in_1,pathway_native_jaccard,pathway_native_tversky_1_in_2,pathway_native_tversky_2_in_1,pathway_inferred_jaccard,pathway_inferred_tversky_1_in_2,pathway_inferred_tversky_2_in_1
0,DB00006,DB06605,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0.0,...,1.000000,0.333333,0.5,0.5,0.0,0.0,0.0,0.090023,0.091954,0.810811
1,DB00006,DB06695,1.0,1.0,1.0,0.111111,0.5,0.125,0.1,0.5,...,1.000000,1.000000,1.0,1.0,0.0,0.0,0.0,1.000000,1.000000,1.000000
2,DB00006,DB01254,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0.0,...,0.090909,0.000000,0.0,0.0,0.0,0.0,0.0,0.216397,0.768582,0.231479
3,DB00006,DB01609,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
4,DB00006,DB01586,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0.0,...,0.222222,0.000000,0.0,0.0,0.0,0.0,0.0,0.034254,0.051341,0.093315
